In [ ]:
from dotenv import load_dotenv
import os

load_dotenv('../.env')
API_KEY = os.getenv('API_KEY')
if not API_KEY or API_KEY == "your api key":
    raise ValueError("API_KEY not found in env file, please set your API KEY there asap for this to work.")

In [15]:
import requests
import pandas as pd

In [30]:
# YouTube URLs
urls = [
    "https://www.youtube.com/watch?v=PghQPGacrlI",  # suv1
    "https://www.youtube.com/watch?v=XRUNyPm9RxI",  # suv2
    "https://www.youtube.com/watch?v=UQGTUonY5qs",  # sedan1
    "https://www.youtube.com/watch?v=OVS0BMMrU5M",  # sedan2
    "https://www.youtube.com/watch?v=7_2WpbOdONg",  # hatchback1
    "https://www.youtube.com/watch?v=sMFqMdBc48M",  # hatchback2
    "https://www.youtube.com/watch?v=avd1GdpB_2A",  # coupe1
    "https://www.youtube.com/watch?v=nMn_ueG80Zw",  # coupe2
    "https://www.youtube.com/watch?v=J-CcwVxcfaQ",  # coupe3
]
categories = ["suv", "sedan", "hatchback", "coupe"]
freq=[2,2,2,3]

# Extract the video IDs
video_ids = [url.split("v=")[-1] for url in urls]

In [31]:
def get_video_title(video_id, API_KEY):
    url = "https://www.googleapis.com/youtube/v3/videos"
    params = {
        "part": "snippet",
        "id": video_id,
        "key": API_KEY
    }
    response = requests.get(url, params=params)
    data = response.json()
    items = data.get("items", [])
    if items:
        return items[0]["snippet"]["title"]
    return "Unknown Title"

In [ ]:
def getCommentsFromVideo(VIDEO_ID, csv_file, API_KEY, category_id = 0):
    rows = []
    video_title = get_video_title(VIDEO_ID, API_KEY)
    url = "https://www.googleapis.com/youtube/v3/commentThreads"

    params = {
        "part": "snippet",
        "videoId": VIDEO_ID,
        "maxResults": 100,
        "key": API_KEY
    }

    while True:
        response = requests.get(url, params=params)
        data = response.json()
        
        for item in data["items"]:
            comment = item["snippet"]["topLevelComment"]["snippet"]

            rows.append({
                "video_id": VIDEO_ID,
                "video_title": video_title,
                "author": comment["authorDisplayName"],
                "comment": comment["textOriginal"],
                "likes": comment["likeCount"],
                "published_at": comment["publishedAt"],
                "category": categories[category_id]
            })

        if "nextPageToken" in data:
            params["pageToken"] = data["nextPageToken"]
        else:
            break

    df = pd.DataFrame(rows)

    if os.path.exists(csv_file):
        df.to_csv(csv_file, mode='a', index=False, header=False)
    else:
        df.to_csv(csv_file, mode='w', index=False, header=True)

    print("Done collecting comments for video:", VIDEO_ID)
    print("Total comments collected:", len(df))

In [33]:
freq_id = 0
for id in video_ids:
    getCommentsFromVideo(id,"..\data\fetched_yt_comments.csv", API_KEY, freq_id)
    freq[freq_id]-=1
    if freq[freq_id] <= 0:
        freq_id+=1
        if freq_id >= len(freq):
            print("category limit reached")
            break
    else:
        continue

Done collecting comments for video: PghQPGacrlI
Total comments collected: 1890
Done collecting comments for video: XRUNyPm9RxI
Total comments collected: 748
Done collecting comments for video: UQGTUonY5qs
Total comments collected: 406
Done collecting comments for video: OVS0BMMrU5M
Total comments collected: 337
Done collecting comments for video: 7_2WpbOdONg
Total comments collected: 59
Done collecting comments for video: sMFqMdBc48M
Total comments collected: 101
Done collecting comments for video: avd1GdpB_2A
Total comments collected: 1166
Done collecting comments for video: nMn_ueG80Zw
Total comments collected: 2579
Done collecting comments for video: J-CcwVxcfaQ
Total comments collected: 1021
category limit reached
